# 02 - Análise de negócio e hipóteses

## Funções e tabelas

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import chi2_contingency, mannwhitneyu

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from olist.config import FIGURES_DIR, INTERIM_DATA_DIR, REPORTS_DIR, ensure_project_dirs
from olist.data import build_order_level_dataset, load_all, load_table
from olist.features import add_delivery_features, add_review_target, add_temporal_features
from olist.plots import save_current_figure, set_plot_style

ensure_project_dirs()
set_plot_style()

In [ ]:
parquet_path = INTERIM_DATA_DIR / 'order_level_dataset.parquet'
csv_path = INTERIM_DATA_DIR / 'order_level_dataset.csv'

if parquet_path.exists():
    order_level = pd.read_parquet(parquet_path)
elif csv_path.exists():
    order_level = pd.read_csv(csv_path, parse_dates=[
        'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
        'order_delivered_customer_date', 'order_estimated_delivery_date'
    ])
else:
    tables = load_all(include_geolocation=False)
    order_level = build_order_level_dataset(tables)

order_level = add_review_target(add_delivery_features(add_temporal_features(order_level)))
delivered = order_level[order_level['order_status'].eq('delivered') & order_level['review_score'].notna()].copy()
print(delivered.shape)

## H1: Pedidos entregues com atraso tem maior chance de review baixo?

Sinal esperado: pedidos atrasados devem ter media de review menor e taxa de `review_score <= 2` maior.

In [ ]:
h1_df = delivered.dropna(subset=['is_late', 'is_low_review']).copy()

h1_summary = (
    h1_df.groupby('is_late')
    .agg(orders=('order_id', 'nunique'), avg_review=('review_score', 'mean'), low_review_rate=('is_low_review', 'mean'), avg_delay_days=('delay_days', 'mean'))
    .reset_index()
)
display(h1_summary)

contingency = pd.crosstab(h1_df['is_late'], h1_df['is_low_review'])
chi2, p_value, dof, expected = chi2_contingency(contingency)
print(f'Chi-square p-value: {p_value:.6f}')

h1_summary['is_late'] = h1_summary['is_late'].map({
    0.0: 'nÃ£o',
    1.0: 'sim'
})
sns.set_theme(style='whitegrid')

fig, ax = plt.subplots(figsize=(7, 5))

sns.barplot(
    data=h1_summary,
    x='is_late',
    y='low_review_rate',
    hue='is_late',
    palette=['#4C78A8', '#E45756'],
    legend=False,
    width=0.6,
    ax=ax
)

ax.set_title(
    'Pedidos Atrasados Recebem Mais Reviews Negativos',
    fontsize=15,
    weight='bold',
    pad=15
)

ax.set_xlabel('Pedido atrasado')
ax.set_ylabel('Taxa de reviews baixos', fontsize=11)
ax.yaxis.set_major_formatter(PercentFormatter(1))

for container in ax.containers:

    labels = [
        f'{bar.get_height()*100:.1f}%'
        for bar in container
    ]

    ax.bar_label(
        container,
        labels=labels,
        padding=4,
        fontsize=11,
        weight='bold'
        
        
    )

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()

save_current_figure(
    FIGURES_DIR / '02_h1_atraso_review_baixo.png'
)

plt.show()

##### Observações

- Pedidos atrasados apresentaram aproximadamente ~54% de probabilidade de receber avaliação ruim, contra cerca de ~9% nos pedidos sem atraso.

- A média das avaliações caiu de aproximadamente 4,29 para 2,57 quando houve atraso na entrega, indicando deterioração expressiva da experiência do cliente.

- O teste Qui-Quadrado retornou `p < 0,001`, indicando associação estatisticamente significativa entre atraso logístico e avaliações negativas.

### Conclusão

Existe forte evidência de associação entre atrasos logísticos e insatisfação dos clientes. Pedidos entregues com atraso apresentam probabilidade significativamente maior de receber avaliações ruins, sugerindo que a experiência de entrega é um dos fatores mais relevantes na percepção negativa do consumidor.

Embora a análise não permita afirmar causalidade direta, a magnitude e consistência do efeito observado tornam o atraso logístico um forte candidato a prioridade operacional.

### O que isso significa para a empresa

- A logística influencia diretamente a experiência do cliente.
- Atrasos podem impactar negativamente reputação, retenção e percepção de qualidade da plataforma.
- Melhorar SLA, previsibilidade e eficiência operacional tende a reduzir avaliações negativas e melhorar a satisfação geral dos consumidores.

## H2: Frete alto em relação ao valor dos produtos prejudica a avaliação?

Frete relativo calculado como `freight_value / products_value`. Para reduzir efeito de outliers, usei winsorização simples pelo percentil 99.
- Aqui não interessante observar o valor do frete sozinho, mas sim a "PERCEPÃÇÃO". Porque:
    1. 20 reais de frente num produto de 500 reais = perceptivelmente justo.
    2. 20 reais de frente num produto de 30 reais = perceptivelmente absurdo.

In [ ]:
h2_df = delivered.dropna(subset=['freight_ratio', 'is_low_review']).copy()
h2_df = h2_df[h2_df['freight_ratio'].between(0, h2_df['freight_ratio'].quantile(0.99))]
h2_df['freight_ratio_bin'] = pd.qcut(h2_df['freight_ratio'], q=4, duplicates='drop', labels = ["baixo", "medio", "alto", "muito_alto"])

h2_summary = (
    h2_df.groupby('freight_ratio_bin')
    .agg(orders=('order_id', 'nunique'), avg_freight_ratio=('freight_ratio', 'mean'), avg_review=('review_score', 'mean'), low_review_rate=('is_low_review', 'mean'))
    .reset_index()
)
display(h2_summary)

low = h2_df.loc[h2_df['is_low_review'].eq(1), 'freight_ratio']
ok = h2_df.loc[h2_df['is_low_review'].eq(0), 'freight_ratio']
stat, p_value = mannwhitneyu(low, ok, alternative='two-sided')
print(f'Mann-Whitney p-value: {p_value:.6f}')

sns.set_theme(style='whitegrid')

fig, ax = plt.subplots(figsize=(8, 5))

sns.lineplot(
    data=h2_summary,
    x='freight_ratio_bin',
    y='low_review_rate',
    marker='o',
    linewidth=2.5,
    markersize=9,
    color='#4C78A8',
    ax=ax
)

ax.set_title(
    'Frete Relativo Mais Alto EstÃ¡ Associado\n'
    'a Maior Taxa de Reviews Negativos',
    fontsize=15,
    weight='bold',
    pad=15
)


ax.set_xlabel(
    'Frete / valor dos produtos',
    fontsize=11
)

ax.set_ylabel(
    'Taxa de reviews baixos',
    fontsize=11
)

ax.yaxis.set_major_formatter(PercentFormatter(1))

ax.grid(axis='y', alpha=0.25)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_ylim(
    h2_summary['low_review_rate'].min() - 0.001,
    h2_summary['low_review_rate'].max() + 0.0015
)

plt.tight_layout()

save_current_figure(
    FIGURES_DIR / '02_h2_frete_relativo.png'
)

plt.show()

##### Observações

- À medida que o frete relativo aumenta, a taxa de reviews negativos também cresce.

- A taxa de avaliações negativas (`low_review_rate`) aumentou de aproximadamente 12,3% para 13,4%, representando um efeito pequeno em magnitude.

- A média das avaliações caiu gradualmente conforme aumentou a proporção entre frete e valor dos produtos.

- O teste de Mann-Whitney retornou `p < 0,001`, indicando diferença estatisticamente significativa entre os grupos analisados.

### Conclusão

Diferentemente do atraso logístico, o frete relativo apresentou associação mais fraca com avaliações negativas. Embora pedidos com maior proporção de frete tenham mostrado leve aumento na incidência de reviews ruins, o efeito observado foi relativamente pequeno em termos práticos.

Isso sugere que os clientes parecem mais sensíveis à qualidade e previsibilidade da entrega do que ao custo proporcional do frete em si.

### O que isso significa para a empresa

- A percepção do cliente não depende apenas do produto ou do prazo de entrega, mas também da percepção de justiça no custo logístico.

- Fretes considerados “caros demais” em relação ao valor comprado podem deteriorar parcialmente a experiência do consumidor.

- Ainda assim, os resultados indicam que melhorias operacionais na entrega provavelmente possuem impacto mais relevante na satisfação do cliente do que reduções marginais no frete proporcional.

## H3: Categorias e estados especí­ficos concentram atrasos e insatisfaçãoo

Aqui priorizo categorias e estados com volume mí­nimo para evitar conclusões Instáveis.

In [ ]:
# Tabela para verificar como o impacto das avaliaÃ§Ãµes interage com a categoria
category_review_summary = (
    delivered.dropna(subset = ['is_late', 'is_low_review'])
    .groupby(['dominant_category', 'is_late',])
    .agg(orders = ('order_id', 'nunique'), avg_review = ('review_score', 'mean'), low_review_rate = ('is_low_review', 'mean'))
    .reset_index()
)
category_review_summary['low_review_rate'] = category_review_summary['low_review_rate'] * 100

category_review_summary.head(20)

######
Observações:
- avg_review cai muito 
- low_review_rate sobe muito (geralmente 40% - 70%)

Conclusão:
- O impacto do atraso é consistente entre as categorias

In [ ]:
# Categorias com baixa volumetria (< 50) foram desconsiderados para evitar interpretaÃ§Ãµes distorcidas por amostras pequenas
valid_categories = (
    category_review_summary.groupby('dominant_category')['orders']
    .min()
    .loc[lambda x: x >= 50]
    .index
)

category_review_summary_filter_50 = category_review_summary[
    category_review_summary['dominant_category'].isin(valid_categories)
]

category_review_summary_filter_50 = category_review_summary_filter_50[category_review_summary_filter_50['is_late'] == 1].sort_values(by = 'low_review_rate')

# Tabela para observar a taxa de atraso por categoria
category_late = (
    delivered.groupby("dominant_category")
      .agg(
          total_orders=("order_id", "count"),
          late_orders=("is_late", "sum")
      )
    .reset_index()
)

category_late["late_rate"] = (
    category_late["late_orders"] /
    category_late["total_orders"]
) * 100

# Tabela com a taxa de criticalidade de cada categoria
category_review_summary_criticality = pd.merge(
    category_review_summary_filter_50,
    category_late[['dominant_category', 'late_rate']],
    how = 'left',
    on = 'dominant_category'
)

category_review_summary_criticality['criticidade'] = (category_review_summary_criticality['low_review_rate'] * category_review_summary_criticality['late_rate']) / 100

category_review_summary_criticality = category_review_summary_criticality.sort_values(by = 'criticidade',ascending = False)

display(category_review_summary_criticality.head(10))


plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=category_review_summary_criticality,
    x="late_rate",
    y="low_review_rate",
    size="orders",
    sizes=(50, 1000),
    alpha=0.7
)

# adicionar nomes das categorias
for _, row in category_review_summary_criticality.iterrows():
    plt.text(
        row["late_rate"] + 0.05,
        row["low_review_rate"] + 0.2,
        row["dominant_category"],
        fontsize=12
    )

plt.xlabel("Taxa de atraso (%)")
plt.ylabel("Taxa de reviews baixos com atraso (%)")
plt.title("Criticidade por categoria")
plt.axvline(category_review_summary_criticality["late_rate"].mean(), ls="--", color="gray")
plt.axhline(category_review_summary_criticality["low_review_rate"].mean(), ls="--", color="gray")
plt.grid(True, alpha=0.3)
save_current_figure(FIGURES_DIR / '02_h3_criticidade_por_categorias.png')

plt.show()

##### Observações

- O atraso logístico se mostrou um forte indicador de insatisfação entre diferentes categorias de produto.

- Algumas categorias aparentam ser significativamente mais sensíveis a falhas de entrega, como:
  - `musical_instruments`
  - `baby`
  - `watches_gifts`

  Nesses casos, os consumidores parecem apresentar menor tolerância a atrasos logísticos.

- Possíveis hipóteses para esse comportamento incluem:
  1. Compras com maior componente emocional;
  2. Maior urgência percebida;
  3. Uso associado a ocasiões específicas ou datas relevantes.

- Embora a análise não permita comprovar causalidade psicológica, o padrão operacional identificado é consistente e relevante.

- Frequência e impacto precisam ser avaliados em conjunto:
  1. A simples existência de atraso não é suficiente para priorizar ações;
  2. A análise de criticidade mostrou que algumas categorias combinam:
     - alta taxa de atraso;
     - e alta probabilidade de review negativo quando o atraso ocorre.
  3. Essas categorias representam maior risco operacional para a experiência do cliente.

- O índice de criticidade utilizado neste estudo deve ser interpretado como uma métrica heurística de priorização operacional, e não como uma métrica estatística formal.

### Conclusão

Algumas categorias concentram risco operacional significativamente maior, combinando atrasos frequentes e forte insatisfação do cliente. Categorias relacionadas a móveis, decoração e utilidades domésticas apresentaram criticidade elevada tanto pelo volume operacional quanto pela incidência de avaliações negativas associadas a atrasos.

### O que isso significa para a empresa

- Nem todas as categorias do marketplace geram o mesmo nível de risco operacional. Existem segmentos claramente mais críticos dentro do catálogo.

- O impacto dos atrasos não é apenas operacional — é também reputacional, afetando percepção de qualidade, satisfação e potencial retenção de clientes.

- A empresa poderia priorizar monitoramento logístico, revisão de SLA e estratégias específicas para categorias com maior criticidade operacional.

In [ ]:
# Tabela para verificar como o impacto das avaliaÃ§Ãµes interage com os estados
state_review_summary = (
    delivered.dropna(subset = ['is_late', 'is_low_review'])
    .groupby(['customer_state', 'is_late',])
    .agg(orders = ('order_id', 'nunique'), avg_review = ('review_score', 'mean'), low_review_rate = ('is_low_review', 'mean'))
    .reset_index()
)
state_review_summary['low_review_rate'] = state_review_summary['low_review_rate'] * 100

state_review_summary

###### Observações:
- avg_review cai muito 
- low_review_rate sobe muito (geralmente ~43% - 68%)

Conclusão:
- O impacto do atraso também é consistente entre os estados

In [ ]:
# Estados com baixa volumetria (< 50) foram desconsiderados para evitar interpretaÃ§Ãµes distorcidas por amostras pequenas
valid_state = (
    state_review_summary.groupby('customer_state')['orders']
    .min()
    .loc[lambda x: x >= 50]
    .index
)

state_review_summary_filter_50 = state_review_summary[
    state_review_summary['customer_state'].isin(valid_state)
]

state_review_summary_filter_50 = state_review_summary_filter_50[state_review_summary_filter_50['is_late'] == 1].sort_values(by = 'low_review_rate')

# Tabela para observar a taxa de atraso por categoria
state_late = (
    delivered.groupby("customer_state")
      .agg(
          total_orders=("order_id", "count"),
          late_orders=("is_late", "sum")
      )
    .reset_index()
)

state_late["late_rate"] = (
    state_late["late_orders"] /
    state_late["total_orders"]
) * 100

# Tabela com a taxa de criticalidade de cada categoria
state_review_summary_criticality = pd.merge(
    state_review_summary_filter_50,
    state_late[['customer_state', 'late_rate']],
    how = 'left',
    on = 'customer_state'
)

state_review_summary_criticality['criticidade'] = (state_review_summary_criticality['low_review_rate'] * state_review_summary_criticality['late_rate']) / 100

state_review_summary_criticality = state_review_summary_criticality.sort_values(by = 'criticidade',ascending = False)

display(state_review_summary_criticality)

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=state_review_summary_criticality,
    x="late_rate",
    y="low_review_rate",
    size="orders",
    sizes=(50, 1000),
    alpha=0.7
)

# adicionar nomes das categorias
for _, row in state_review_summary_criticality.iterrows():
    plt.text(
        row["late_rate"] + 0.05,
        row["low_review_rate"] + 0.2,
        row["customer_state"],
        fontsize=14
    )

plt.xlabel("Taxa de atraso (%)")
plt.ylabel("Taxa de reviews baixos com atraso (%)")
plt.title("Criticidade por estado")
plt.axvline(state_review_summary_criticality["late_rate"].mean(), ls="--", color="gray")
plt.axhline(state_review_summary_criticality["low_review_rate"].mean(), ls="--", color="gray")
plt.grid(True, alpha=0.3)
save_current_figure(FIGURES_DIR / '02_h3_criticidade_por_estados.png')

plt.show()

###### 
Observações:
- O impacto operacional do atraso varia significativamente entre segmentos e regiões, permitindo priorização estratégica baseada em frequência, severidade e volume.
- Estados do Nordeste concentram as maiores taxas de atraso e criticidade operacional:
    1. AL apresentou a maior criticidade:
       1. ~23% de atraso
       2. ~61% de reviews negativos em pedidos atrasados
    2. MA, CE, PI, SE também apresentaram:
       1. atraso acima da média
       2. elevada taxa de avaliações negativas
       3. baixa média de review
    3. RJ se destacou pelo alto volume de pedidos atrasados aliado a forte insatisfação:
       1. mais de 1600 pedidos atrasados
       2. ~67% de reviews negativos em pedidos atrasados.
       3. grande número absoluto de clientes impactados
       4. maior risco reputacional agregado
       5. possível prioridade estratégica.
    4. Estados do Sul e Sudeste, especialmente SP, PR, MG apresentaram:
       1. menores taxas de atraso
       2. menores níveis de criticidade
       3. melhores médias de avaliação
       4. Mas apenas RELATIVO, pois o impacto negativo ainda é alto, reforçando que atraso é negativo em qualquer contexto

Conclusão:
- A criticidade logística não está distribuída de forma homogênea entre os estados. Regiões do Nordeste apresentaram simultaneamente maiores taxas de atraso e maior incidência de avaliações negativas, sugerindo desafios logísticos regionais mais intensos. Já estados do Sul e Sudeste demonstraram operação mais estável e menor impacto na satisfação do cliente.

O que isso significa para a empresa:
- Gargalos logísticos regionais
- Diferenças operacionais relevantes entre estados.
- Impacto desigual na experiência do cliente
- A operação logística parece funcionar muito melhor em algumas regiões do que em outras.

## H4: Pedidos com maior complexidade tem pior experiência.
Complexidade é aproximada por quantidade de itens, produtos unicos, vendedores unicos e métodos de pagamento.

In [ ]:
complexity_cols = ['order_items', 'unique_products', 'unique_sellers', 'payment_methods']
complexity_summary = []

for col in complexity_cols:
    tmp = delivered[[col, 'is_low_review', 'review_score']].dropna().copy()
    tmp[f'{col}_bin'] = pd.qcut(tmp[col].rank(method='first'), q=4, duplicates='drop', labels = ["muito_baixa", "baixa", "media", "alta"])
    summary = tmp.groupby(f'{col}_bin').agg(avg_value=(col, 'mean'), low_review_rate=('is_low_review', 'mean'), avg_review=('review_score', 'mean'), orders=(col, 'size')).reset_index()
    summary['feature'] = col
    complexity_summary.append(summary)

complexity_summary = pd.concat(complexity_summary, ignore_index=True)
display(complexity_summary)

sns.set_theme(style='whitegrid')

fig, ax = plt.subplots(figsize=(10, 6))

sns.lineplot(
    data=complexity_summary,
    x='avg_value',
    y='low_review_rate',
    hue='feature',
    marker='o',
    linewidth=2.5,
    markersize=8,
    palette=['#4C78A8', '#E45756', '#72B7B2', '#F58518'],
    ax=ax
)

ax.set_title(
    'Produtos Mais Complexos Tendem a Receber\n'
    'Mais Reviews Negativos',
    fontsize=16,
    weight='bold',
    pad=18
)


ax.set_xlabel(
    'Valor mÃ©dio da faixa',
    fontsize=11
)

ax.set_ylabel(
    'Taxa de reviews negativos',
    fontsize=11
)


ax.yaxis.set_major_formatter(PercentFormatter(1))


ax.grid(axis='y', alpha=0.25)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend = ax.legend(
    title='Proxy de complexidade',
    frameon=False
)

legend.get_title().set_fontsize(11)


plt.tight_layout()

save_current_figure(
    FIGURES_DIR / '02_h4_complexidade.png'
)

plt.show()

######
Observações:
- A variável que realmente parece aumentar a insatisfação é `order_items` (quantidade total de itens no pedido). Pedidos com mais itens tendem a receber significativamente mais reviews negativos.
    1. Quando há apenas 1 item, a taxa de review ruim está em ~11%. Já com 1,56 itens, a taxa sobe para ~17%, representando um aumento relativamente expressivo.
    2. A média das avaliações caiu de ~4,2 para ~3,9.

- A variável `unique_products` (quantidade de produtos distintos) também sugere deterioração da experiência do cliente.
    1. Quando há apenas 1 produto único, a taxa de review ruim está em ~11,7%. Já com 1,15 produtos únicos, essa taxa sobe para ~15%.

- O mesmo comportamento aparece na variável `unique_sellers` (quantidade de vendedores).
    1. Com apenas 1 vendedor, a taxa de reviews negativos é de ~12%, mas um leve aumento na complexidade (1,05 vendedores) já eleva essa taxa para ~14%.

- Somente a variável `payment_methods` (múltiplos métodos de pagamento) apresentou efeito fraco ou praticamente inexistente.

- Portanto: nem toda complexidade operacional possui o mesmo impacto.

- A maioria das médias ficou próxima de 1. Isso revela que a maior parte dos pedidos da Olist é extremamente simples, normalmente contendo:
    1. 1 item;
    2. 1 produto;
    3. 1 vendedor;
    4. 1 método de pagamento.

- Isso sugere que pequenos aumentos de complexidade já são suficientes para gerar impactos perceptíveis na experiência do cliente.

Conclusão:
- Indicadores de complexidade logística, especialmente quantidade de itens, produtos e vendedores por pedido, apresentaram associação consistente com aumento de avaliações negativas. O resultado sugere que pedidos operacionalmente mais complexos tendem a gerar maior fricção na experiência do cliente.

O que isso significa para a empresa:
- Monitorar pedidos multi-item
- Melhorar coordenação entre vendedores
- Revisar logística de pedidos compostos
- Criar tratamento operacional prioritário para pedidos mais complexos

## Priorizacao de vendedores

Esta analise ajuda seller success a identificar vendedores com volume relevante e pior experiencia media. Ela deve ser usada com cuidado, pois nem todo problema e responsabilidade direta do vendedor.

In [ ]:
items = load_table('order_items')
seller_orders = items[['order_id', 'seller_id']].drop_duplicates()

seller_risk = (
    seller_orders.merge(delivered[['order_id', 'review_score', 'is_low_review', 'is_late']], on='order_id', how='inner')
    .groupby('seller_id')
    .agg(orders=('order_id', 'nunique'), avg_review=('review_score', 'mean'), late_rate=('is_late', 'mean'), low_review_rate=('is_low_review', 'mean'))
    .query('orders >= 50')
    .sort_values(['low_review_rate', 'orders'], ascending=[False, False])
    .head(20)
    .reset_index()
)
display(seller_risk)